In [6]:
import pandas as pd
import os
import sys
import json
from pathlib import Path
import numpy as np

# Notebook is in notebooks/, so repo root is parent
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

In [7]:
import importlib
import src.models.lstm as l

importlib.reload(l)

<module 'src.models.lstm' from '/Users/brandonng/Documents/GitHub/ClinicalDigitalTwin/src/models/lstm.py'>

In [8]:
from src.models.lstm import load_data_files

# Get repo root relative to the current notebook
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Load static preprocessing config
config_path = os.path.join(repo_root, "configs", "lstm_pipeline_params.json")
with open(config_path, "r") as f:
    config = json.load(f)

# Set input and output directories
in_dir = os.path.join(repo_root, config["paths"]["in_dir"])


ed_vitals, clinical_encounters, ecg_records = load_data_files(in_dir, config)

In [9]:
ed_vitals.head()

,subject_id,stay_id,charttime,temperature,heartrate,resprate,o2sat,sbp,dbp,chartdate
0,17195991,38649090,2110-01-11 21:45:00,NaN,141.0,24.0,94.0,NaN,NaN,2110-01-11
1,17195991,38649090,2110-01-11 21:50:00,NaN,123.0,24.0,91.0,151.0,120.0,2110-01-11
2,17195991,38649090,2110-01-11 22:00:00,NaN,133.0,23.0,99.0,180.0,86.0,2110-01-11
3,17195991,38649090,2110-01-11 22:07:00,NaN,164.0,24.0,99.0,198.0,116.0,2110-01-11
4,17195991,38649090,2110-01-11 22:23:00,NaN,130.0,16.0,100.0,235.0,126.0,2110-01-11


In [10]:
clinical_encounters.head()

,subject_id,hadm_id,ed_stay_id,ed_intime,ed_outtime,hosp_admittime,hosp_dischtime,race,gender,anchor_age,death_time,icu_stay_id,icu_intime,icu_outtime,icu_count,diagnosis,icd_codes,diagnosis_labels,is_cardiovascular
0,10000032,22595853.0,33258284.0,2180-05-06 19:17:00,2180-05-06 23:30:00,2180-05-06 22:23:00,2180-05-07 17:15:00,WHITE,F,52.0,2180-09-09 00:00:00,NaN,NaN,NaN,NaN,"['Portal hypertension', 'Other ascites', 'Cirr...","['572.3', '789.59', '571.5', '070.70', '496', ...",[],False
1,10000032,22841357.0,38112554.0,2180-06-26 15:54:00,2180-06-26 21:31:00,2180-06-26 18:27:00,2180-06-27 18:49:00,WHITE,F,52.0,2180-09-09 00:00:00,NaN,NaN,NaN,NaN,['Unspecified viral hepatitis C with hepatic c...,"['070.71', '789.59', '287.5', '276.1', '496', ...",[],False
2,10000032,25742920.0,35968195.0,2180-08-05 20:58:00,2180-08-06 01:44:00,2180-08-05 23:44:00,2180-08-07 17:50:00,WHITE,F,52.0,2180-09-09 00:00:00,NaN,NaN,NaN,NaN,['Chronic hepatitis C without mention of hepat...,"['070.54', '789.59', 'V46.2', '571.5', '276.7'...",[],False
3,10000032,29079034.0,39399961.0,2180-07-23 05:54:00,2180-07-23 14:00:00,2180-07-23 12:35:00,2180-07-25 17:55:00,WHITE,F,52.0,2180-09-09 00:00:00,[39553978],['2180-07-23 14:00:00'],['2180-07-23 23:50:47'],1.0,"['Other iatrogenic hypotension', 'Chronic hepa...","['458.29', '070.44', '799.4', '276.1', '789.59...",[],False
4,10000117,27988844.0,33176849.0,2183-09-18 08:41:00,2183-09-18 20:20:00,2183-09-18 18:10:00,2183-09-21 16:30:00,WHITE,F,48.0,NaN,NaN,NaN,NaN,NaN,['Unspecified intracapsular fracture of left f...,"['S72.012A', 'W01.0XXA', 'Y93.K1', 'Y92.480', ...",['valvular_endocardial_disease'],True


In [16]:
ecg_records.head()

,subject_id,study_id,file_name,ecg_time,path,hadm_id,ed_stay_id,icu_stay_id,in_hosp,in_ed,...,rr_interval,p_onset,p_end,qrs_onset,qrs_end,t_end,p_axis,qrs_axis,t_axis,full_report
0,16598616,40000035,40000035,2126-04-01 19:32:00,files/p1659/p16598616/s40000035/40000035,26101986.0,32731561.0,NaN,0,1,...,800,40,162,194,286,576,48,14,33,"['abnormal_ecg', 'infarct_pattern', 'normal_ec..."
1,18370366,40000084,40000084,2179-08-30 11:58:00,files/p1837/p18370366/s40000084/40000084,NaN,37736561.0,NaN,0,1,...,1176,40,140,202,280,602,59,56,45,"['normal_ecg', 'sinus_bradycardia', 'technical..."
2,12576058,40000115,40000115,2163-04-17 16:45:00,files/p1257/p12576058/s40000115/40000115,NaN,36567247.0,NaN,0,1,...,779,305,29999,503,627,971,0,83,50,['unspecified_ecg']
3,14691089,40000143,40000143,2175-07-21 01:01:00,files/p1469/p14691089/s40000143/40000143,23760732.0,38981935.0,NaN,0,1,...,690,348,29999,499,591,890,26,-49,59,"['infarct_pattern', 'sinus_rhythm']"
4,16089780,40000152,40000152,2137-01-29 19:08:00,files/p1608/p16089780/s40000152/40000152,NaN,31326149.0,NaN,0,1,...,645,40,124,176,268,536,51,29,63,"['borderline_ecg', 'sinus_rhythm', 'st_t_abnor..."


In [18]:
print(ecg_records.shape)
print(ecg_records['in_ed'].value_counts())

(403121, 26)
in_ed
0    244837
1    158284
Name: count, dtype: int64


In [13]:
seq_lengths = (
    ed_vitals.groupby(["subject_id", "stay_id"])
    .size()
    .reset_index(name="n_vitals")
)
print(seq_lengths["n_vitals"].describe(percentiles=[.5, .75, .90, .95, .99]))

count    291592.000000
mean          4.201895
std           3.521600
min           1.000000
50%           3.000000
75%           5.000000
90%           8.000000
95%          11.000000
99%          17.000000
max         109.000000
Name: n_vitals, dtype: float64


In [20]:
ecg_counts = (
    ecg_records[ecg_records['in_ed'] == 1].groupby("subject_id")
    .size()
    .reset_index(name="n_ecgs")
)
print(ecg_counts["n_ecgs"].describe(percentiles=[.5, .75, .90, .95, .99]))
print(f"Patients with >8 ECGs: {(ecg_counts['n_ecgs'] > 8).mean():.1%}")

count    75583.000000
mean         2.094175
std          2.727339
min          1.000000
50%          1.000000
75%          2.000000
90%          4.000000
95%          6.000000
99%         12.000000
max        154.000000
Name: n_ecgs, dtype: float64
Patients with >8 ECGs: 2.3%
